In [27]:
import glob
import os
import random
import numpy as np
import csv
import json

In [28]:
class MajorityClassifier:
    def fit(self, X, y):
        counts = {}
        for label in y:
            counts[label] = counts.get(label, 0) + 1
        self.most_frequent = max(counts, key=counts.get)
        return self
 
    def predict(self, X):
        return [self.most_frequent for _ in X]   

In [29]:
def train_test_split(X, y, test_size=0.2, seed=42):
    idx = list(range(len(y)))
    random.Random(seed).shuffle(idx)
    n_test = int(len(y) * test_size)
    test, train = idx[:n_test], idx[n_test:]
    return ([X[i] for i in train], [X[i] for i in test],
            [y[i] for i in train], [y[i] for i in test])

In [30]:
def get_accuracy(y_true, y_pred):
    correct = sum(1 for t, p in zip(y_true, y_pred) if t == p)
    return correct / len(y_true)

In [31]:
DATA_ROOT = "/path/to/autoPET"
 
# Build labels, AI helped with this snippet
X, y = [], []
for path in sorted(glob.glob(os.path.join(DATA_ROOT, "labelsTr", "*.nii.gz"))):
    mask = nib.load(path).dataobj
    X.append(os.path.basename(path))
    y.append(1 if mask[...].any() else 0)
 
print(f"Scans: {len(y)}  |  tumor: {sum(y)}  |  no tumor: {len(y) - sum(y)}") # AI for print

Scans: 0  |  tumor: 0  |  no tumor: 0


In [32]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
#clf = MajorityClassifier().fit(X_train, y_train)
#print(f"Most frequent class: {clf.most_frequent}") # AI for print
#y_pred = clf.predict(X_test)
#print(f"Baseline Accuracy: {get_accuracy(y_test, y_pred):.4f}") # AI for print

In [33]:
# Load FDG labels (AI helped with this)
labels = {}
with open("data/fdg_metadata.csv") as f:
    for row in csv.DictReader(f):
        if row["Modality"] == "PT":
            patient = row["Subject ID"].replace("PETCT_", "")
            study = row["File Location"].split("/")[-2]
            name = f"fdg_{patient}_{study}"
            labels[name] = 0 if row["diagnosis"] == "NEGATIVE" else 1

# Load the first training/validation split
with open("data/splits_final.json") as f:
    split = json.load(f)[0]

X_train = [x for x in split["train"] if x in labels]
X_test = [x for x in split["val"] if x in labels]

y_train = [labels[x] for x in X_train]
y_test = [labels[x] for x in X_test]

# Test existing classifier
model = MajorityClassifier().fit(X_train, y_train)
predictions = model.predict(X_test)

print("Training scans:", len(X_train))
print("Testing scans:", len(X_test))
print("Accuracy:", get_accuracy(y_test, predictions))

Training scans: 822
Testing scans: 192
Accuracy: 0.4375


In [ ]:
#Binary classification components. no encoder or pretrained weights included

import torch
from torch import nn


class BinaryClassificationHead(nn.Module):
    #Map [B, C, D, H, W] features (or pooled [B, C]) to [B, 1] logits.

    def __init__(self, in_features: int):
        super().__init__()
        if in_features <= 0:
            raise ValueError("in_features must be positive")
        self.pool = nn.AdaptiveAvgPool3d(1)
        self.fc = nn.Linear(in_features, 1)

    def forward(self, features: torch.Tensor) -> torch.Tensor:
        if not isinstance(features, torch.Tensor):
            raise TypeError("Expected a feature tensor; adapt the encoder output explicitly")
        if features.ndim not in (2, 5):
            raise ValueError("Expected [B, C] or [B, C, D, H, W] features")
        if features.shape[1] != self.fc.in_features:
            raise ValueError("Feature channels must match the configured in_features")
        if features.ndim == 5:
            features = self.pool(features).flatten(1)
        return self.fc(features)